<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/ecfr_landing_success_10_minutes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ECFR Aircraft Landing Success Analysis: 10-Minute Window

This notebook parses an ECFR aircraft-engine text export and determines whether each `LANDING_STATISTICS` event has supporting evidence within a configurable 10-minute window.

### Evidence evaluated
- Direct landing records in `LANDING_STATISTICS`
- Fault records in `FF_ATTRIBUTES`
- Maintenance-computer records in `MC_ATTRIBUTES` and `MC_HISTORY_ATTRIBUTES`
- Shutdown evidence in `ROLLDOWN`
- Flight-cycle reconciliation from `USAGE_STATISTICS`
- Contradictory or cautionary evidence from `ENGINE_REDLINE`, `RECORD_FAILURES`, and `TYPE0_THRESHOLD`

### Decision interpretation
- **SUCCESSFUL_CONFIRMED_WITHIN_WINDOW**: Direct landing event plus supporting event within 600 seconds.
- **LANDING_RECORDED_WINDOW_NOT_MET**: Direct landing event exists, but no supporting timestamp is found within 600 seconds.
- **REVIEW_REQUIRED**: A fault, maintenance-computer event, redline, or record failure appears close to the landing.

> This notebook confirms engine-system recognition of landing completion. It does not certify touchdown quality, hard landing, runway performance, or airworthiness. Manufacturer-approved data and airframe parameters are required for operational conclusions.


In [ ]:

# Configuration
from pathlib import Path
from datetime import datetime, timedelta
import io, re, math, json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

WINDOW_SECONDS = 600
INPUT_FILE = None  # Set a local path here, or leave None to upload in Google Colab.
OUTPUT_PREFIX = "ecfr_landing_analysis"
print(f"Landing corroboration window: {WINDOW_SECONDS} seconds")


Landing corroboration window: 600 seconds


In [ ]:

# Upload the ECFR text file in Google Colab, or use a local file when running elsewhere.
def obtain_input_file(configured_path=None):
    if configured_path:
        p = Path(configured_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        name, raw = next(iter(uploaded.items()))
        p = Path(name)
        p.write_bytes(raw)
        return p
    except ImportError:
        candidates = sorted(Path('.').glob('*.txt'))
        if not candidates:
            raise FileNotFoundError("Place the ECFR .txt file in the working directory or set INPUT_FILE.")
        return candidates[0]

input_path = obtain_input_file(INPUT_FILE)
print("Input file:", input_path)

In [ ]:

# Generic section parser for tab-delimited ECFR exports.
# A section begins with an uppercase identifier, followed by a tab-delimited header row.
SECTION_RE = re.compile(r"^[A-Z][A-Z0-9_ ]*$")

def clean_line(line):
    return line.rstrip('\r\n')

def is_section_name(line):
    s=line.strip()
    return bool(s and '\t' not in s and SECTION_RE.fullmatch(s))

def make_unique(columns):
    seen={}; out=[]
    for i,c in enumerate(columns):
        c=(c or f"column_{i+1}").strip()
        n=seen.get(c,0); seen[c]=n+1
        out.append(c if n==0 else f"{c}_{n+1}")
    return out

def parse_ecfr_sections(path):
    lines=Path(path).read_text(encoding='utf-8', errors='replace').splitlines()
    sections={}; i=0
    while i < len(lines):
        if not is_section_name(lines[i]):
            i+=1; continue
        name=lines[i].strip(); i+=1
        if i>=len(lines): break
        # Skip empty lines before the section header.
        while i<len(lines) and not lines[i].strip(): i+=1
        if i>=len(lines) or '\t' not in lines[i]:
            sections[name]=pd.DataFrame(); continue
        headers=make_unique(lines[i].rstrip('\t').split('\t')); i+=1
        records=[]
        while i<len(lines) and not is_section_name(lines[i]):
            line=lines[i]; i+=1
            if not line.strip(): continue
            vals=line.rstrip('\t').split('\t')
            if len(vals)<len(headers): vals += ['']*(len(headers)-len(vals))
            if len(vals)>len(headers): vals=vals[:len(headers)]
            records.append(vals)
        sections[name]=pd.DataFrame(records,columns=headers)
    return sections

sections=parse_ecfr_sections(input_path)
print(f"Parsed {len(sections)} sections")
for key in ['LANDING_STATISTICS','FF_ATTRIBUTES','MC_ATTRIBUTES','MC_HISTORY_ATTRIBUTES','ROLLDOWN','USAGE_STATISTICS','ENGINE_REDLINE','RECORD_FAILURES','TYPE0_THRESHOLD']:
    print(f"{key:24s}: {len(sections.get(key,pd.DataFrame()))} rows")


In [ ]:

# Timestamp and numeric helpers
def to_number(v):
    try:
        if v is None or str(v).strip()=="": return np.nan
        return float(str(v).strip())
    except Exception:
        return np.nan

def timestamp_from_fields(date_value, time_value):
    """Parse YYMMDD plus HHMMSS. Handles time values with omitted leading zeroes."""
    try:
        d=str(int(float(date_value))).zfill(6)
        t=str(int(float(time_value))).zfill(6)
        if d=='000000' or t=='000000': return pd.NaT
        return pd.Timestamp(datetime.strptime(d+t,'%y%m%d%H%M%S'))
    except Exception:
        return pd.NaT

def first_col(df, candidates):
    lookup={c.lower().strip():c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup: return lookup[candidate.lower()]
    return None

def add_timestamp(df, date_candidates, time_candidates, output='timestamp'):
    df=df.copy()
    dc=first_col(df,date_candidates); tc=first_col(df,time_candidates)
    df[output]=[timestamp_from_fields(d,t) for d,t in zip(df[dc],df[tc])] if dc and tc else pd.NaT
    return df

def nearest_event(landing_row, event_df, event_name, window_seconds, allow_before=True, allow_after=True):
    if event_df is None or event_df.empty or 'timestamp' not in event_df or pd.isna(landing_row['timestamp']):
        return None
    valid=event_df.dropna(subset=['timestamp']).copy()
    if valid.empty: return None
    valid['delta_seconds']=(valid['timestamp']-landing_row['timestamp']).dt.total_seconds()
    mask=pd.Series(True,index=valid.index)
    if not allow_before: mask &= valid['delta_seconds']>=0
    if not allow_after: mask &= valid['delta_seconds']<=0
    valid=valid[mask]
    if valid.empty: return None
    valid['absolute_delta']=valid['delta_seconds'].abs()
    row=valid.sort_values('absolute_delta').iloc[0]
    result=row.to_dict(); result['source_section']=event_name
    result['within_window']=bool(row['absolute_delta']<=window_seconds)
    return result


In [ ]:

# Normalize the relevant sections and create timestamps.
landing=sections.get('LANDING_STATISTICS',pd.DataFrame()).copy()
if landing.empty:
    raise ValueError('LANDING_STATISTICS was not found or contains no rows.')
landing=add_timestamp(landing,['Landing GMT Date'],['Landing GMT Time'])

rolldown=add_timestamp(sections.get('ROLLDOWN',pd.DataFrame()),['Rolldown GMT Date'],['Rolldown GMT Time'])
ff=add_timestamp(sections.get('FF_ATTRIBUTES',pd.DataFrame()),['FF GMT Date'],['FF GMT Time'])

mc_frames=[]
for sec in ['MC_ATTRIBUTES','MC_HISTORY_ATTRIBUTES']:
    df=sections.get(sec,pd.DataFrame())
    if df.empty: continue
    # Prefer channel 0, then unsuffixed fields.
    x=add_timestamp(df,['MC GMT Date 0','MC GMT Date'],['MC GMT Time 0','MC GMT Time'])
    x['source_mc_section']=sec
    mc_frames.append(x)
mc=pd.concat(mc_frames,ignore_index=True,sort=False) if mc_frames else pd.DataFrame()

redline=add_timestamp(sections.get('ENGINE_REDLINE',pd.DataFrame()),
                      ['Exceedance GMT Date','Engine Redline GMT Date','GMT Date'],
                      ['Exceedance GMT Time','Engine Redline GMT Time','GMT Time'])

print('Landing timestamp range:',landing['timestamp'].min(),'to',landing['timestamp'].max())
print('Valid timestamp counts:',{'landing':landing.timestamp.notna().sum(),'rolldown':rolldown.get('timestamp',pd.Series(dtype='datetime64[ns]')).notna().sum(),'ff':ff.get('timestamp',pd.Series(dtype='datetime64[ns]')).notna().sum(),'mc':mc.get('timestamp',pd.Series(dtype='datetime64[ns]')).notna().sum(),'redline':redline.get('timestamp',pd.Series(dtype='datetime64[ns]')).notna().sum()})


In [ ]:

# Decision engine
# Direct LANDING_STATISTICS is primary evidence. Supporting evidence is searched in +/- 10 minutes,
# except ROLLDOWN, which must occur after the landing.
def value_from(row, names, default=np.nan):
    for n in names:
        if n in row and str(row[n]).strip()!='': return row[n]
    return default

results=[]
for idx,lrow in landing.iterrows():
    rd=nearest_event(lrow,rolldown,'ROLLDOWN',WINDOW_SECONDS,allow_before=False,allow_after=True)
    ff_n=nearest_event(lrow,ff,'FF_ATTRIBUTES',WINDOW_SECONDS)
    mc_n=nearest_event(lrow,mc,'MC_ATTRIBUTES/MC_HISTORY_ATTRIBUTES',WINDOW_SECONDS)
    rl=nearest_event(lrow,redline,'ENGINE_REDLINE',WINDOW_SECONDS)

    supporting=[]; caution=[]
    if rd and rd['within_window']: supporting.append('ROLLDOWN')
    if mc_n and mc_n['within_window']:
        # MC is evidence of contemporaneous state; ground flags are surfaced separately.
        supporting.append('MC_TIMESTAMP')
    if ff_n and ff_n['within_window']: caution.append('FF_FAULT_NEAR_LANDING')
    if rl and rl['within_window']: caution.append('ENGINE_REDLINE_NEAR_LANDING')

    direct=True
    if direct and supporting and not caution:
        decision='SUCCESSFUL_CONFIRMED_WITHIN_WINDOW'
    elif direct and supporting and caution:
        decision='SUCCESSFUL_REVIEW_REQUIRED'
    elif direct and caution:
        decision='LANDING_RECORDED_REVIEW_REQUIRED'
    else:
        decision='LANDING_RECORDED_WINDOW_NOT_MET'

    results.append({
        'landing_sequence':idx+1,
        'landing_timestamp':lrow['timestamp'],
        'landing_ecu_time':to_number(value_from(lrow,['Landing ECU Operating Time'])),
        'aircraft_serial':value_from(lrow,['Landing Aircraft S/N']),
        'latitude':to_number(value_from(lrow,['Landing Latitude'])),
        'longitude':to_number(value_from(lrow,['Landing Longitude'])),
        'flight_hours':to_number(value_from(lrow,['Landing Flight Hours','Landing Flight Duration'])),
        'nearest_rolldown_timestamp':rd.get('timestamp') if rd else pd.NaT,
        'rolldown_delta_seconds':rd.get('delta_seconds') if rd else np.nan,
        'rolldown_within_10_min':bool(rd and rd['within_window']),
        'nearest_ff_timestamp':ff_n.get('timestamp') if ff_n else pd.NaT,
        'ff_delta_seconds':ff_n.get('delta_seconds') if ff_n else np.nan,
        'ff_within_10_min':bool(ff_n and ff_n['within_window']),
        'nearest_mc_timestamp':mc_n.get('timestamp') if mc_n else pd.NaT,
        'mc_delta_seconds':mc_n.get('delta_seconds') if mc_n else np.nan,
        'mc_within_10_min':bool(mc_n and mc_n['within_window']),
        'nearest_redline_timestamp':rl.get('timestamp') if rl else pd.NaT,
        'redline_within_10_min':bool(rl and rl['within_window']),
        'supporting_evidence':', '.join(supporting),
        'caution_flags':', '.join(caution),
        'decision':decision
    })
result=pd.DataFrame(results)
display(result.tail(10))


In [ ]:

# Cross-check usage counters and data-integrity sections.
def section_as_records(name):
    d=sections.get(name,pd.DataFrame()).copy()
    return d

usage=section_as_records('USAGE_STATISTICS')
record_failures=section_as_records('RECORD_FAILURES')
threshold=section_as_records('TYPE0_THRESHOLD')

print('Usage statistics:')
display(usage)
print('Record failures:',len(record_failures))
if not threshold.empty:
    print('Threshold summary:')
    display(threshold)


In [ ]:

# Detailed verdict for the latest recorded landing.
latest=result.sort_values('landing_timestamp').iloc[-1]
window_end=latest['landing_timestamp']+pd.Timedelta(seconds=WINDOW_SECONDS)
print('='*72)
print('LATEST LANDING VERDICT')
print('='*72)
print('Landing timestamp           :',latest['landing_timestamp'])
print('10-minute window end        :',window_end)
print('Decision                    :',latest['decision'])
print('Nearest rolldown            :',latest['nearest_rolldown_timestamp'])
print('Rolldown elapsed seconds    :',latest['rolldown_delta_seconds'])
print('Rolldown within 600 seconds :',latest['rolldown_within_10_min'])
print('FF event within 600 seconds :',latest['ff_within_10_min'])
print('MC event within 600 seconds :',latest['mc_within_10_min'])
print('Redline within 600 seconds  :',latest['redline_within_10_min'])
print('Caution flags               :',latest['caution_flags'] or 'None')
print()
if latest['decision']=='SUCCESSFUL_CONFIRMED_WITHIN_WINDOW':
    print('Conclusion: landing was directly recorded and corroborated within 10 minutes.')
elif latest['decision']=='LANDING_RECORDED_WINDOW_NOT_MET':
    print('Conclusion: landing was directly recorded, but the strict 10-minute corroboration rule was not met.')
else:
    print('Conclusion: landing was recorded, but nearby caution evidence requires review.')


In [ ]:

# Export analysis files for download.
result.to_csv(f'{OUTPUT_PREFIX}_events.csv',index=False)
with pd.ExcelWriter(f'{OUTPUT_PREFIX}_report.xlsx',engine='openpyxl') as writer:
    result.to_excel(writer,sheet_name='Landing_Decisions',index=False)
    landing.to_excel(writer,sheet_name='Landing_Statistics',index=False)
    rolldown.to_excel(writer,sheet_name='Rolldown',index=False)
    ff.to_excel(writer,sheet_name='FF_Attributes',index=False)
    mc.to_excel(writer,sheet_name='MC_Attributes',index=False)
    usage.to_excel(writer,sheet_name='Usage_Statistics',index=False)
    redline.to_excel(writer,sheet_name='Engine_Redline',index=False)

print('Created:')
print(f'  {OUTPUT_PREFIX}_events.csv')
print(f'  {OUTPUT_PREFIX}_report.xlsx')

try:
    from google.colab import files
    # Uncomment either line to download automatically after execution.
    # files.download(f'{OUTPUT_PREFIX}_events.csv')
    # files.download(f'{OUTPUT_PREFIX}_report.xlsx')
except ImportError:
    pass



## Notes for adapting the rule

- Change `WINDOW_SECONDS = 600` to alter the correlation window.
- The current rule treats `LANDING_STATISTICS` as direct evidence of a recorded landing.
- `ROLLDOWN` is accepted only after the landing timestamp.
- `FF_ATTRIBUTES` and `ENGINE_REDLINE` are caution signals, not automatic proof of landing failure.
- `MC_ATTRIBUTES` is treated as contextual evidence. Add manufacturer-specific decoding for `MC Flight Ground 0/1`, status words, and maintenance codes before assigning stronger meaning.
- For hard-landing assessment, add airframe data such as weight-on-wheels, vertical acceleration, radio altitude, sink rate, pitch, roll, groundspeed, brake status, and thrust-reverser state.
